# Google Colabに搭載されたgemini apiを使ってみる

API_KEYなどなくても、text generationが使えます。

In [ ]:
# @title List available models
from google.colab import ai

ai.list_models()

In [ ]:
from google.colab import ai

response = ai.generate_text("What is the capital of France?", model_name="google/gemini-3.5-flash")
print(response)

In [ ]:
import pandas as pd

df = pd.read_csv('http://s3.amazonaws.com/assets.datacamp.com/course/Kaggle/train.csv')
df.head()

In [ ]:
def ask_csv(data, query):
  prompt = f"""
  あなたは優秀なデータサイエンティストです。
  以下のタイタニックデータに対して、質問={query}に答えなさい。
  <data>
  {data}
  </data>"""

  response = ai.generate_text(prompt, model_name="google/gemini-3.5-flash")
  return response


In [ ]:
response=ask_csv(df.to_csv(),"whats the square root of the average age?")
print(response)

In [ ]:
query=""" make three age groupes:  age <15, 15=<age<60<, age>=60. draw pie chart with legend for these age groupe,
          what is the most survived group? what is the opposite?
"""
response=ask_csv(df.to_csv(),query)
print(response)

## おまけ
ai.generate_text以外にも、 Geminiを使う方法があります。例えば、pydanticというライブラリと結合したものを以下では使ってます。

In [ ]:
!pip install -q --no-warn-conflicts "colab-ai-bridge[pydantic-ai] @ git+https://github.com/drillan/colab-ai-bridge"

In [ ]:
from colab_ai_bridge.pydantic_ai import ColabPydanticAIModel
from pydantic_ai import Agent

# モデルの作成（セットアップは自動実行される）
model = ColabPydanticAIModel()
agent = Agent(model)

result = agent.run_sync("フランスの首都は？")
print(result.output)

Pydanticは、geminの出力を、予め設定したデータ構造で出力させることができます。このように構造をもった出力をさせることを「構造化出力」と呼ばれています。

In [ ]:
from pydantic import BaseModel

class CityInfo(BaseModel):
    name: str
    country: str
    population: int
    famous_landmarks: list[str]

# モデルの作成
model = ColabPydanticAIModel("google/gemini-2.5-flash")
agent = Agent(model, output_type=CityInfo)

# 構造化された出力を取得
result = agent.run_sync("東京の情報を教えてください")
print(f"都市名: {result.output.name}")
print(f"国: {result.output.country}")
print(f"人口: {result.output.population:,}人")
print(f"有名な観光地: {', '.join(result.output.famous_landmarks)}")

In [ ]:
result